# Exploratory data analysis - UCI Online Retail

EDA, cleaning and RFM features only. The modelling pipeline (churn model, simulated campaign, T-learner, Qini evaluation) lives in `simulate.py`.

Download `Online Retail.xlsx` from https://archive.ics.uci.edu/dataset/352/online+retail into `data/raw/` and run from the repository root.


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)


In [ ]:
file_path = "data/raw/Online Retail.xlsx"

df = pd.read_excel(file_path)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df.isnull().sum()


In [ ]:
df['CustomerID'].nunique()


In [ ]:

df[df['Quantity'] < 0].shape, df[df['UnitPrice'] <= 0].shape


In [ ]:
df[df['Quantity'] < 0].head(10)


In [ ]:
df['CustomerID'].isna().mean(), df['CustomerID'].isna().sum()


In [ ]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df = df.sort_values('InvoiceDate')
df.head()


In [ ]:
n_rows = len(df)
n_customers = df['CustomerID'].nunique()
n_invoices = df['InvoiceNo'].nunique()
total_revenue = df['TotalPrice'].sum()

n_rows, n_customers, n_invoices, total_revenue


In [ ]:
daily_revenue = (
    df.groupby(df['InvoiceDate'].dt.date)['TotalPrice']
      .sum()
      .reset_index(name='Revenue')
)

daily_revenue.head()


In [ ]:
plt.figure(figsize=(14,5))
plt.plot(daily_revenue['InvoiceDate'], daily_revenue['Revenue'])
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.title('Daily Revenue Over Time')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
country_revenue = (
    df.groupby('Country')['TotalPrice']
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

country_revenue.head(10)


In [ ]:
top_countries = country_revenue.head(10)

plt.figure(figsize=(10,5))
sns.barplot(data=top_countries, x='Country', y='TotalPrice')
plt.xticks(rotation=45)
plt.title('Revenue by Country (Top 10)')
plt.ylabel('Total Revenue')
plt.tight_layout()
plt.show()


In [ ]:
customer_revenue = (
    df.groupby('CustomerID')['TotalPrice']
      .sum()
      .reset_index(name='CustomerRevenue')
)

customer_revenue.describe()


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(customer_revenue['CustomerRevenue'], bins=50)
plt.xscale('log')
plt.xlabel('Customer Revenue (log scale)')
plt.title('Distribution of Revenue per Customer')
plt.tight_layout()
plt.show()


In [ ]:
orders_per_customer = (
    df.groupby('CustomerID')['InvoiceNo']
      .nunique()
      .reset_index(name='OrderCount')
)

orders_per_customer.describe()


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(orders_per_customer['OrderCount'], bins=30)
plt.title("Distribution of Number of Orders per Customer")
plt.xlabel("Order Count")
plt.ylabel("Count")
plt.show()


In [ ]:
one_time_customers = (orders_per_customer['OrderCount'] == 1).sum()
repeat_customers = (orders_per_customer['OrderCount'] > 1).sum()

one_time_customers, repeat_customers


In [ ]:
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

monthly_orders = df.groupby('InvoiceMonth')['InvoiceNo'].nunique()

plt.figure(figsize=(12,5))
monthly_orders.plot()
plt.title("Number of Orders Per Month")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)
plt.show()


In [ ]:
basket_size = df.groupby('InvoiceNo')['Quantity'].sum()

plt.figure(figsize=(8,5))
sns.histplot(basket_size, bins=50)
plt.title("Distribution of Basket Size (Items per Invoice)")
plt.xlabel("Items per Order")
plt.ylabel("Count")
plt.show()


In [ ]:
df = df[df['CustomerID'].notnull()]


In [ ]:
df = df[df['Quantity'] > 0]


In [ ]:
df = df[df['TotalPrice'] > 0]


In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


In [ ]:
df = df.sort_values('InvoiceDate')


In [ ]:
df.info(), df.describe()


In [ ]:
import datetime as dt

snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)
snapshot_date


In [ ]:
rfm = (
    df.groupby('CustomerID')
    .agg({
        'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
        'InvoiceNo': 'nunique',                                   # Frequency
        'TotalPrice': 'sum'                                       # Monetary
    })
    .rename(columns={
        'InvoiceDate': 'Recency',
        'InvoiceNo': 'Frequency',
        'TotalPrice': 'Monetary'
    })
)

rfm.head()


In [ ]:
rfm.describe()


In [ ]:
rfm.hist(figsize=(12,6), bins=50)
plt.tight_layout()
plt.show()
